# 03 - nn.Module, Parameters & Buffers

## Learning Objectives

- Understand `nn.Module` as the base class for all PyTorch models
- Define custom modules with `__init__` (register layers) and `forward` (define computation)
- Use `nn.Linear`, `nn.ReLU`, and `nn.Sequential` to build networks
- Inspect model parameters with `.parameters()` and `.named_parameters()`
- Register non-learnable state with `register_buffer`
- Save and load models using `state_dict()`

## Prerequisites

- Notebook 01 (Tensors, Shapes, Dtypes & Broadcasting)
- Notebook 02 (Autograd and the Computation Graph)
- Basic Python OOP (classes, inheritance)
- PyTorch installed (`pip install torch`)

## Table of Contents

1. [nn.Module: The Base Class](#1-nnmodule-the-base-class)
2. [Defining a Custom Module](#2-defining-a-custom-module)
3. [nn.Linear, nn.ReLU, nn.Sequential](#3-nnlinear-nnrelu-nnsequential)
4. [Building an MLP (Class-Based)](#4-building-an-mlp-class-based)
5. [Building an MLP (nn.Sequential)](#5-building-an-mlp-nnsequential)
6. [Inspecting Parameters](#6-inspecting-parameters)
7. [Buffers: Non-Learnable State](#7-buffers-non-learnable-state)
8. [state_dict: Saving and Loading Models](#8-state_dict-saving-and-loading-models)
9. [Exercises](#9-exercises)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. nn.Module: The Base Class

Every neural network in PyTorch inherits from `torch.nn.Module`. It provides:

| Feature | Description |
|---|---|
| Parameter management | Automatically tracks all learnable parameters |
| Device movement | `.to(device)` moves all parameters at once |
| Mode switching | `.train()` / `.eval()` toggle training/inference behaviour |
| Serialization | `.state_dict()` for saving/loading weights |
| Hooks | Register forward/backward hooks for debugging |

To define a custom model you must:
1. Subclass `nn.Module`
2. Call `super().__init__()` in `__init__`
3. Define layers as attributes in `__init__` (so they are registered)
4. Implement `forward(self, x)` to define the computation

In [ ]:
# Minimal example: a single linear layer wrapped in nn.Module
class MinimalModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()  # MUST call this
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


torch.manual_seed(42)
model = MinimalModel(3, 2)

# The model is callable -- calling it invokes forward()
x = torch.randn(4, 3)  # batch of 4, 3 features each
output = model(x)       # DO NOT call model.forward(x) directly
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output:\n{output}")

## 2. Defining a Custom Module

The `__init__` / `forward` pattern is the foundation of all PyTorch model building.

- **`__init__`**: register all layers, parameters, and buffers
- **`forward`**: define how data flows through those layers

In [ ]:
class TwoLayerNet(nn.Module):
    """A simple two-layer neural network."""

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # Register layers as attributes -- this is how nn.Module
        # discovers them and tracks their parameters.
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Define the computation graph
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


torch.manual_seed(42)
net = TwoLayerNet(input_dim=4, hidden_dim=8, output_dim=2)
print(net)

# Test with dummy input
x = torch.randn(5, 4)  # batch of 5, 4 features
out = net(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")

In [ ]:
# Common mistake: defining a layer but NOT as an attribute
class BrokenModel(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # This layer is NOT registered because it's a local variable
        layer = nn.Linear(in_dim, out_dim)
        # nn.Module cannot see `layer` -- it has no parameters!

    def forward(self, x):
        # This would fail because `layer` is not accessible here
        pass

broken = BrokenModel(4, 2)
print(f"BrokenModel parameters: {list(broken.parameters())}")
print("  -> Empty! The Linear layer was not registered.")

# Fix: assign it as self.layer = nn.Linear(...)

## 3. nn.Linear, nn.ReLU, nn.Sequential

### nn.Linear

A fully connected (dense) layer: $y = xW^T + b$

| Parameter | Shape |
|---|---|
| `weight` | `(out_features, in_features)` |
| `bias` | `(out_features,)` |

### nn.ReLU

The Rectified Linear Unit activation: $\text{ReLU}(x) = \max(0, x)$

- No learnable parameters
- Can also use `torch.relu(x)` as a function instead of a module

### nn.Sequential

A container that chains modules in order. Input flows through each module sequentially.

In [ ]:
# nn.Linear in detail
torch.manual_seed(42)
linear = nn.Linear(in_features=3, out_features=5)

print(f"Weight shape: {linear.weight.shape}")  # (5, 3)
print(f"Bias shape:   {linear.bias.shape}")    # (5,)
print(f"\nWeight:\n{linear.weight}")
print(f"\nBias:\n{linear.bias}")

# Forward pass: y = x @ W^T + b
x = torch.randn(2, 3)  # batch of 2, 3 features
y = linear(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {y.shape}")

In [ ]:
# nn.Linear without bias
torch.manual_seed(42)
linear_no_bias = nn.Linear(3, 5, bias=False)
print(f"Has bias: {linear_no_bias.bias is not None}")
print(f"Num parameters: {sum(p.numel() for p in linear_no_bias.parameters())}")

In [ ]:
# nn.ReLU as a module vs as a function
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

# Module form
relu_module = nn.ReLU()
print(f"Module: {relu_module(x)}")

# Functional form
import torch.nn.functional as F
print(f"Function: {F.relu(x)}")

# Both produce the same result
print(f"Equal: {torch.equal(relu_module(x), F.relu(x))}")

In [ ]:
# nn.Sequential: chain modules in order
torch.manual_seed(42)

seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3),
)

print(seq_model)

# Forward pass: input flows through each module in order
x = torch.randn(2, 4)
out = seq_model(x)
print(f"\nOutput shape: {out.shape}")

In [ ]:
# nn.Sequential with named modules (OrderedDict)
from collections import OrderedDict

torch.manual_seed(42)
named_model = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(4, 8)),
    ('relu1', nn.ReLU()),
    ('fc2', nn.Linear(8, 3)),
]))

print(named_model)
print(f"\nAccess by name: {named_model.fc1}")
print(f"Access by index: {named_model[0]}")

## 4. Building an MLP (Class-Based)

A Multi-Layer Perceptron (MLP) is a stack of linear layers with non-linear activations between them.

In [ ]:
class MLP(nn.Module):
    """A 3-layer MLP for classification."""

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # no activation on final layer (raw logits)
        return x


torch.manual_seed(42)
mlp = MLP(input_dim=784, hidden_dim=128, output_dim=10)
print(mlp)

# Test with MNIST-like input
x = torch.randn(32, 784)  # batch of 32 flattened 28x28 images
logits = mlp(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {logits.shape}")
print(f"Logits (first sample): {logits[0]}")

In [ ]:
# Count total parameters
total_params = sum(p.numel() for p in mlp.parameters())
trainable_params = sum(p.numel() for p in mlp.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Breakdown per layer
print("\nPer-layer breakdown:")
print(f"  fc1: {784*128} (weight) + {128} (bias) = {784*128 + 128:,}")
print(f"  fc2: {128*128} (weight) + {128} (bias) = {128*128 + 128:,}")
print(f"  fc3: {128*10} (weight) + {10} (bias) = {128*10 + 10:,}")
print(f"  Total: {784*128 + 128 + 128*128 + 128 + 128*10 + 10:,}")

## 5. Building an MLP (nn.Sequential)

For simple feed-forward architectures, `nn.Sequential` is more concise.

In [ ]:
# Same MLP as above, using nn.Sequential
torch.manual_seed(42)

mlp_seq = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print(mlp_seq)

# Test
x = torch.randn(32, 784)
logits_seq = mlp_seq(x)
print(f"\nOutput shape: {logits_seq.shape}")

In [ ]:
# When to use class-based vs Sequential:
#
# nn.Sequential:
#   - Simple feed-forward architectures
#   - No branching, skip connections, or custom logic
#   - Quick prototyping
#
# Class-based (nn.Module subclass):
#   - Skip/residual connections
#   - Multiple inputs or outputs
#   - Conditional logic in forward pass
#   - Custom behaviour (e.g., returning intermediate features)

print("nn.Sequential is great for simple stacks.")
print("Subclass nn.Module when you need custom forward logic.")

## 6. Inspecting Parameters

PyTorch provides several methods to inspect model parameters:

| Method | Returns |
|---|---|
| `.parameters()` | Iterator over all `Parameter` tensors |
| `.named_parameters()` | Iterator of `(name, Parameter)` tuples |
| `.children()` | Iterator over direct sub-modules |
| `.named_children()` | Iterator of `(name, Module)` tuples |
| `.modules()` | Iterator over all modules (recursive) |
| `.named_modules()` | Iterator of `(name, Module)` tuples (recursive) |

In [ ]:
torch.manual_seed(42)
mlp = MLP(input_dim=784, hidden_dim=128, output_dim=10)

# .parameters() -- just the tensors
print("=== .parameters() ===")
for i, p in enumerate(mlp.parameters()):
    print(f"  param {i}: shape={p.shape}, requires_grad={p.requires_grad}")

In [ ]:
# .named_parameters() -- names + tensors
print("=== .named_parameters() ===")
for name, p in mlp.named_parameters():
    print(f"  {name:15s} shape={str(p.shape):15s} numel={p.numel():>6,}")

In [ ]:
# .children() and .named_children() -- direct sub-modules
print("=== .named_children() ===")
for name, child in mlp.named_children():
    print(f"  {name}: {child}")

In [ ]:
# .modules() and .named_modules() -- all modules recursively
print("=== .named_modules() ===")
for name, mod in mlp.named_modules():
    print(f"  {name if name else '(root)':15s} -> {mod.__class__.__name__}")

In [ ]:
# Practical: print a parameter summary table
def print_model_summary(model):
    """Print a summary of model parameters."""
    print(f"{'Name':25s} {'Shape':20s} {'# Params':>10s}")
    print("-" * 58)
    total = 0
    for name, p in model.named_parameters():
        n = p.numel()
        total += n
        print(f"{name:25s} {str(list(p.shape)):20s} {n:>10,}")
    print("-" * 58)
    print(f"{'TOTAL':25s} {'':20s} {total:>10,}")


print_model_summary(mlp)

## 7. Buffers: Non-Learnable State

Sometimes a module needs to store state that is **not a learnable parameter** (no gradients), but should still:
- Be saved/loaded with `state_dict()`
- Move to the correct device with `.to(device)`

Examples:
- Running mean/variance in BatchNorm
- A fixed positional encoding
- A step counter

Use `self.register_buffer(name, tensor)` to register such state.

In [ ]:
class NormalizingLayer(nn.Module):
    """A module that normalizes input using pre-computed mean and std.
    
    mean and std are NOT learnable -- they are buffers.
    """

    def __init__(self, num_features, mean=None, std=None):
        super().__init__()
        if mean is None:
            mean = torch.zeros(num_features)
        if std is None:
            std = torch.ones(num_features)

        # Register as buffers -- NOT parameters
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, x):
        return (x - self.mean) / self.std


norm = NormalizingLayer(
    num_features=3,
    mean=torch.tensor([0.485, 0.456, 0.406]),
    std=torch.tensor([0.229, 0.224, 0.225]),
)

# Buffers appear in state_dict but NOT in parameters
print("Parameters:", list(norm.parameters()))  # empty!
print("\nstate_dict:")
for k, v in norm.state_dict().items():
    print(f"  {k}: {v}")

In [ ]:
# Buffers move with .to(device) just like parameters
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
norm = norm.to(device)

print(f"mean device: {norm.mean.device}")
print(f"std device:  {norm.std.device}")

# Test forward pass
x = torch.randn(4, 3, device=device)
out = norm(x)
print(f"\nInput:\n{x}")
print(f"\nNormalized:\n{out}")

In [ ]:
# Buffer vs parameter vs plain attribute
class ComparisonModule(nn.Module):
    def __init__(self):
        super().__init__()
        # Parameter: learnable, in state_dict, moves with .to()
        self.weight = nn.Parameter(torch.randn(3))

        # Buffer: NOT learnable, in state_dict, moves with .to()
        self.register_buffer('running_mean', torch.zeros(3))

        # Plain attribute: NOT in state_dict, does NOT move with .to()
        self.some_constant = torch.ones(3)

    def forward(self, x):
        return x * self.weight + self.running_mean


comp = ComparisonModule()

print("Parameters (learnable):")
for name, p in comp.named_parameters():
    print(f"  {name}: {p.shape}")

print("\nBuffers (non-learnable, tracked):")
for name, b in comp.named_buffers():
    print(f"  {name}: {b.shape}")

print("\nstate_dict (parameters + buffers):")
for k in comp.state_dict():
    print(f"  {k}")

print(f"\nPlain attribute 'some_constant' is NOT in state_dict.")
print(f"  'some_constant' in state_dict: {'some_constant' in comp.state_dict()}")

## 8. state_dict: Saving and Loading Models

A `state_dict` is an `OrderedDict` mapping parameter/buffer names to their tensor values.

**Recommended approach:** Save and load the `state_dict`, not the entire model object.

| Method | Saves | Use Case |
|---|---|---|
| `torch.save(model.state_dict(), path)` | Weights only | Recommended |
| `torch.save(model, path)` | Entire model (uses pickle) | Fragile, not recommended |

In [ ]:
# Inspect a state_dict
torch.manual_seed(42)
mlp = MLP(input_dim=784, hidden_dim=128, output_dim=10)

sd = mlp.state_dict()
print("Keys in state_dict:")
for key in sd:
    print(f"  {key:15s} shape={sd[key].shape}")

In [ ]:
import tempfile
import os

# Save state_dict
torch.manual_seed(42)
mlp_save = MLP(input_dim=10, hidden_dim=16, output_dim=3)

# Get a prediction before saving
x_test = torch.randn(2, 10)
pred_before = mlp_save(x_test)
print(f"Prediction before save:\n{pred_before}")

# Save to a temporary file
save_path = os.path.join(tempfile.gettempdir(), 'mlp_weights.pth')
torch.save(mlp_save.state_dict(), save_path)
print(f"\nSaved to: {save_path}")
print(f"File size: {os.path.getsize(save_path):,} bytes")

In [ ]:
# Load state_dict into a NEW model instance
mlp_load = MLP(input_dim=10, hidden_dim=16, output_dim=3)  # fresh model

# Before loading -- random weights, different predictions
pred_random = mlp_load(x_test)
print(f"Before loading (random weights):\n{pred_random}")

# Load saved weights
mlp_load.load_state_dict(torch.load(save_path, weights_only=True))

# After loading -- same predictions as original model
pred_after = mlp_load(x_test)
print(f"\nAfter loading:\n{pred_after}")

# Verify they match
print(f"\nPredictions match: {torch.allclose(pred_before, pred_after)}")

In [ ]:
# Clean up temporary file
os.remove(save_path)
print("Temporary file cleaned up.")

## 9. Exercises

### Exercise 1: Create a Custom Module with a Skip Connection

Implement a `ResidualBlock` module that:
1. Takes input `x` of shape `(batch, dim)`
2. Passes it through: Linear -> ReLU -> Linear
3. **Adds the original input** to the output (skip connection)
4. Applies a final ReLU

$$\text{output} = \text{ReLU}(x + \text{Linear}_2(\text{ReLU}(\text{Linear}_1(x))))$$

Note: For the skip connection to work, input and output dimensions must match.

In [ ]:
# TODO: Implement ResidualBlock

# class ResidualBlock(nn.Module):
#     def __init__(self, dim):
#         super().__init__()
#         ...
#
#     def forward(self, x):
#         ...


# Test:
# torch.manual_seed(42)
# block = ResidualBlock(dim=64)
# x = torch.randn(8, 64)
# out = block(x)
# print(f"Input shape:  {x.shape}")
# print(f"Output shape: {out.shape}")
# assert out.shape == x.shape, "Output shape should match input shape"

### Exercise 2: Build a Network with Buffers

Create a `ScaledMLP` module that:
1. Has a standard MLP (2 hidden layers)
2. Stores `input_mean` and `input_std` as **buffers**
3. In `forward`, normalizes the input before passing it through the MLP
4. Verify that saving and loading `state_dict` preserves the buffers

In [ ]:
# TODO: Implement ScaledMLP

# class ScaledMLP(nn.Module):
#     def __init__(self, input_dim, hidden_dim, output_dim, input_mean, input_std):
#         super().__init__()
#         ...
#
#     def forward(self, x):
#         x = (x - self.input_mean) / self.input_std
#         ...


# Test:
# torch.manual_seed(42)
# mean = torch.tensor([1.0, 2.0, 3.0])
# std = torch.tensor([0.5, 0.5, 0.5])
# model = ScaledMLP(3, 16, 2, mean, std)
# print(model.state_dict().keys())  # should include input_mean, input_std

### Solutions

In [ ]:
# Solution 1: ResidualBlock
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x                          # save input for skip connection
        out = self.relu(self.fc1(x))           # Linear -> ReLU
        out = self.fc2(out)                    # Linear
        out = self.relu(out + identity)        # add skip connection + ReLU
        return out


torch.manual_seed(42)
block = ResidualBlock(dim=64)
x = torch.randn(8, 64)
out = block(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, "Output shape should match input shape"
print("Skip connection works correctly!")

# Print parameter count
n_params = sum(p.numel() for p in block.parameters())
print(f"Parameters: {n_params:,}")

In [ ]:
# Solution 2: ScaledMLP
class ScaledMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, input_mean, input_std):
        super().__init__()
        # Register normalisation stats as buffers
        self.register_buffer('input_mean', input_mean)
        self.register_buffer('input_std', input_std)

        # MLP layers
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        x = (x - self.input_mean) / self.input_std
        return self.net(x)


torch.manual_seed(42)
mean = torch.tensor([1.0, 2.0, 3.0])
std = torch.tensor([0.5, 0.5, 0.5])
scaled_model = ScaledMLP(3, 16, 2, mean, std)

print("state_dict keys:")
for k in scaled_model.state_dict():
    print(f"  {k}")

# Verify buffers are in state_dict
assert 'input_mean' in scaled_model.state_dict()
assert 'input_std' in scaled_model.state_dict()
print("\nBuffers correctly included in state_dict!")

# Test forward pass
x_test = torch.randn(4, 3) + mean  # data centred around mean
out_test = scaled_model(x_test)
print(f"\nInput shape:  {x_test.shape}")
print(f"Output shape: {out_test.shape}")

## 10. Common Mistakes & Debugging Tips

1. **Forgetting `super().__init__()`**: All parameter tracking breaks. You will get `AttributeError` when trying to assign `nn.Module` attributes.

2. **Not assigning layers as attributes**: If you create `nn.Linear(...)` inside `__init__` but store it in a local variable instead of `self.layer = ...`, it will not be registered and `.parameters()` will miss it.

3. **Calling `model.forward(x)` directly**: Always call `model(x)`. The `__call__` method runs hooks and other important logic before calling `forward`.

4. **Applying softmax before `CrossEntropyLoss`**: `nn.CrossEntropyLoss` already applies softmax internally. Adding it in your `forward` method causes double softmax.

5. **Mismatched dimensions in skip connections**: For residual/skip connections, the input and output dimensions of the skip path must match. Use a projection layer (1x1 linear) if they differ.

6. **Using plain tensors instead of buffers for non-learnable state**: Plain tensor attributes will not be saved in `state_dict()` and will not move with `.to(device)`. Use `register_buffer` instead.

7. **Saving the entire model with `torch.save(model, ...)`**: This uses pickle, which is fragile and breaks when you refactor code. Always save `model.state_dict()` instead.

8. **Loading state_dict with mismatched architecture**: `load_state_dict` will raise an error if the model architecture does not match the saved weights. Use `strict=False` carefully if needed.